In [2]:
%reload_ext autoreload
%autoreload 2

import sys
sys.path.append(r"./../../广东农商_code/")

import os
import pandas as pd 
import numpy as np  
import math  
import matplotlib.pyplot as plt
import copy
import seaborn as sns
from pylab import mpl
from pypmml import Model
mpl.rcParams['font.sans-serif'] = ['SimHei']
mpl.rcParams['axes.unicode_minus'] = False  

FILE_PATH = 'E:/Workspace/lff/20211220-20211224_广州农商/report/'
DATA_PATH = 'E:/Workspace/lff/20211220-20211224_广州农商/data/'

isExists=os.path.exists(FILE_PATH)
if not isExists:
    os.makedirs(FILE_PATH) 

import warnings
warnings.filterwarnings("ignore")

### 授信模型1额度

#### 收入负责授信基准
收入负债授信基准=月收入*可支配比例-月负债

In [3]:
def credit_base_line(income, liability, discretionary_ratio=0.7):
    if income in [0, -1.0, -999]:
        income = 10
    if liability in [-1.0, -999]:
        liability = 0
    bins = np.array([1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32, 33,34,35,36,37,38,39,
                     40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57])
    amount = [0,500,1000,1500,2000,2500,3000,3500,4000,4500,5000,5500,6000,6500,7000,7500,8000,8500,9000,9500,10000,15000,20000,
             25000,30000,35000,40000,45000,50000, 55000, 60000, 65000, 70000, 75000, 80000,85000, 90000, 95000, 100000,200000,300000,
              400000,500000,600000,700000,800000,900000,1000000,2000000,3000000,4000000,5000000,6000000,7000000,8000000,9000000,
             10000000,10000000]
    return amount[np.digitize(income, bins)] * discretionary_ratio - amount[np.digitize(liability, bins)] 

#### 风险调整系数 

In [4]:
def risk_adjusted_factor(score):
    bins = np.array([0, 500, 675, 700, 725, 750, 775, 875, 1000])
    line = [0.8, 0.8, 0.9, 1.0, 1.2, 1.3, 1.4, 1.5, 1.5, 1.5]
    return line[np.digitize(score, bins)]

#### 优质客群系数

In [5]:
def high_quality_factor(data):
    bins = np.array([0, 8, 18, 28, 32])
    line = [1.0, 1.0, 1.2, 1.5, 1.8, 2.0]
    if data['max_loan_offer_amount_level_m12'] == -999:
        return line[0]
    return line[np.digitize(data['max_loan_offer_amount_level_m12'], bins)]

#### 优质客群额度下限

In [6]:
def high_quality_lower(factor):
    bins = np.array([1.0, 1.2, 1.5, 1.8, 2.0, 3])
    line = [0, 0, 3000, 5000, 8000, 10000, 15000, 20000]
    return line[np.digitize(factor, bins)]

#### max(收入负责授信基准 * 期数 * 风险调整系数 * 优质客群系数，优质客群额度下限)

In [62]:
def credit_model_1(data, credit_score, periods=3):
    base = credit_base_line(data.tzbg_debit_card_stab_month_in_amt_m12, data.tzbg_credit_card_instalment_amt_m12, discretionary_ratio=0.7)
    print('credit_base_line: ', base)
    risk_adjusted_customers_factor = risk_adjusted_factor(credit_score)
    print('risk_adjusted_customers_factor: ', risk_adjusted_customers_factor)
    high_quality_customers_factor = high_quality_factor(data)
    print('high_quality_customers_factor: ', high_quality_customers_factor)
    high_quality_customers_credit_line_lower = high_quality_lower(high_quality_customers_factor)
    print('high_quality_customers_credit_line_lower: ', high_quality_customers_credit_line_lower)
    return max(base * periods * risk_adjusted_customers_factor * high_quality_customers_factor, high_quality_customers_credit_line_lower)

### 授信模块2额度  

#### 评分基准额度

In [8]:
def base_line(score):
    bins = np.array([0, 500, 675, 700, 725, 750, 775, 875, 1000])
    line = [5000,5000,10000,14000,18000,23000,25000,28000,30000,35000]
#      line = [3000,3000,5000,8000,12000,15000,18000,20000,25000,40000]
#     line = [5000,5000,10000,15000,20000,25000,35000,40000,45000,50000]
#     line = [i*3 for i in [2500,2500,3500,4500,6000,7500,9000,9500,10500,10500]]
    return line[np.digitize(score, bins)]

#### 优质条件命中系数
1.最近12月借贷类发放金额最大数(等级) > 2w		
2.最近24月借贷类逾期提醒事件累计次数 == 0 或( 最近24月借贷类逾期已还时长最大数等级 < D 且 最近12月借贷类逾期提醒事件累计次数 == 0）		

In [9]:
def high_quality_condition_hit_factor(data, factor=[0.6, 0.8, 1.0, 1.5, 2.0]):
    count = 0
    if data.max_loan_offer_amount_level_m12 >= 27:
        count += 1
    if data.overdue_sum_m24 == 0 or (data.overdue_sum_m12 == 0 and data.max_overdue_repay_delay_level_m24 < 4):
        count += 1
    if data.tzbg_debit_card_total_bal_amt > 23:
        count += 1
    if data.repay_fail_sum_m24 == 0 or (data.repay_fail_sum_m12 == 0 and data.repay_fail_sum_m24 < 4):
        count += 1
    return factor[count]

#### 多头系数
1.最近6月借贷类申请平台累计数>=6 或 最近6月借贷类申请事件累计次数>=15		
2.根据身份证统计，非银行近6个月内贷款申请平台数>=7		

In [10]:
def long_factor(data, factor = 0.4):
    count = 0
    if data.apply_request_count_m6 >= 6 or data.apply_request_sum_m6 >= 15:
        count += 1
    if data.tzbg_loan_nonbank_apply_plt_cnt_m6 >= 7:
        count += 1
    return 1 if count == 0 else factor

#### 逾期系数
1.最近24月借贷类逾期已还时长最大数等级 >= E 或 最近12月借贷类逾期提醒事件累计次数 > 2

In [11]:
def overdue_factor(data, factor = 0.57):
    count = 0
    if data.max_overdue_repay_delay_level_m24 >= 5 or data.overdue_sum_m12 > 2:
        count += 1
    return 1 if count == 0 else factor

#### 评分基准额度 * 优质条件命中系数 * 多头系数 * 逾期系数 * 额度使用率系数

In [12]:
def credit_model_2(data, credit_score):
    base = base_line(credit_score)
#     print('base_line: ', base)
    high_quality_customers_factor = high_quality_condition_hit_factor(data)
#     print('high_quality_customers_factor: ', high_quality_customers_factor)
    long_costomers_factor = long_factor(data)
#     print('long_factor: ', long_costomers_factor)
    overdue_costomers_factor = overdue_factor(data)
#     print('overdue_factor: ', overdue_costomers_factor)
    line_usage_rate = 1
#     print('line_usage_rate: ', line_usage_rate)
    return base * high_quality_customers_factor * long_costomers_factor * overdue_costomers_factor * line_usage_rate

### 信用评分上/下限

In [60]:
def credit_score_bounds(score):
                     
    bins = np.array([0, 500, 675, 700, 725, 750, 775, 875, 1000])
    bounds = [[0, 8000],[0, 12000],[0, 18000],[0, 25000],[0, 30000],[1500, 35000], [3000, 40000],[5000, 80000],[8000, 120000],[8000, 120000]]
    return bounds[np.digitize(score, bins)][0], bounds[np.digitize(score, bins)][1]

In [61]:
credit_score_bounds(769.5294877367635)

(3000, 40000)

### 最终授信
#### min(max(max(授信模型1额度，授信模块2额度),信用评分下限),信用评分上限)

In [2]:
def credit_model(data):
    credit_score = data['APP_SCORE']
#     print('credit_score: ',credit_score)
    credit_line1, credit_line2 = credit_model_1(data, credit_score, periods=1), credit_model_2(data, credit_score)
    print('credit_line1: ',credit_line1)
    print('credit_line2: ',credit_line2)
    credit_score_lower, credit_score_upper = credit_score_bounds(credit_score)
    print('credit_score_lower: ',credit_score_lower)
    print('credit_score_upper: ',credit_score_upper)
    print(min(max(max(credit_line1, credit_line2), credit_score_lower), credit_score_upper))
    print('- - - - - - - - - - - - - - - - - ')
    return min(max(max(credit_line1, credit_line2), credit_score_lower), credit_score_upper)

### 授信额度取整

In [16]:
def limit2int(x):
    if x.STR_LIMIT_init <= 3000:
        return 3000
    elif x.STR_LIMIT_init > 80000:
        return 80000
    else:
        (x.STR_LIMIT_init // 500 + 1) * 500

## 样本测试

In [1]:
data_all = pd.read_pickle(DATA_PATH + 'data_end.pkl')

### 申请评分

In [ ]:
import joblib
model = joblib.load('model.model')
ft_lst = pd.read_csv('keep_lst.csv')['var_names'].to_list()

In [ ]:
def proba2score(prob, pdo=50, rate=2, base_odds=35, base_score=900):
    factor = pdo / np.log(rate)
    offset = base_score - factor * np.log(base_odds)
    return factor * (np.log(1 - prob) - np.log(prob)) + offset

In [ ]:
data_all['proba'] = model.predict_proba(data_all[keep_lst])[:, 1]
data_all['score'] = proba2score(data_end['proba'])

### 计算额度

#### min(max(max(授信模型1额度，授信模块2额度),信用评分下限),信用评分上限)

In [1]:
data_all['STR_LIMIT_init'] = data_all.apply(lambda x: credit_model(x), axis=1)
data_all['STR_LIMIT'] = data_all.apply(lambda x: limit2int(x), axis=1)

In [ ]:
import random
data_all['STR_LIMIT_init'] = [random.randint(3000,80000) for i in range(total)]
data_all['STR_LIMIT'] = data_all.apply(lambda x: limit2int(x), axis=1)
data_all['第6个mob实际账单透支余额'] = [random.randint(1000,50000) for i in range(total)]
data_all['实际额度'] = data_all['第6个mob实际账单透支余额'] + [random.randint(1000,10000) for i in range(total)]

#### 额度分析

In [ ]:
score = data_all['STR_LIMIT_init']
%matplotlib inline

fig, axes = plt.subplots(1,1)
plt.style.use('ggplot')
sns.distplot(score,bins=20,kde=False)

fig.set_figwidth(10)
fig.set_figheight(5)

In [ ]:
score = data_all['STR_LIMIT']

%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1,1)
plt.style.use('ggplot')

sns.distplot(score,bins=10,kde=False)

fig.set_figwidth(10)
fig.set_figheight(5)

In [ ]:
result = pd.DataFrame()
delta = 5000
for i in range(1,20):
    temp = data_all[(data_all['STR_LIMIT_init'] >= i*delta)&(data_all['STR_LIMIT_init']< (i+1)*delta)]
    result = result.append({"分数区间": str(i * delta)+"-"+str(i * delta + delta), 
                           "count": temp.shape[0],
                           '占比':temp.shape[0]/data_all.shape[0]},ignore_index=True)
    
result[["分数区间","count",'占比']]

In [ ]:
result = pd.DataFrame()
delta = 5000
for i in range(0,10):
    temp = data_all[(data_all['STR_LIMIT'] > i * delta)&(data_all['STR_LIMIT'] <= (i+1) * delta)]
    result = result.append({"分数区间": str(i * delta) + "-" + str(i * delta + delta), 
                           "count": temp.shape[0],
                           '占比':temp.shape[0]/data_all.shape[0]},ignore_index=True)
    
result[["分数区间","count",'占比']]

### 审批通过率
基于贷前标卡申请客户的申请数据、人行数据进行贷前风险策略模型的计算   
- 审批通过率=申请件中通过数/(通过数+拒绝数） 

In [ ]:
total = data_all.shape[0]
threshold = 750
data_all['APP_SCORE'] = data_all['score']
data_all['STR_RESULT'] = data_all.apply(lambda x: '通过' if x.score >= threshold else '拒绝', axis=1)
tg = data_all[data_all['STR_RESULT'] == '通过'].shape[0]
jj = data_all[data_all['STR_RESULT'] == '拒绝'].shape[0]
print('通过数', tg)
print('拒绝数', jj)
print('审批通过率', tg/(tg+jj))

### 按建议额度的金额不良率下降比例
即按建议额度的金额坏账率下降度，建议额度即POC策略输出的建议授信额度；统计群体：行里实际审批结果为通过

- （1）额度使用率=第6个mob实际账单透支余额/实际额度；
- （2）按建议额度的金额坏账率=∑(6个月内曾经M2+客户的额度使用率 *建议额度)/∑(客户额度使用率 *建议额度）
- （3）按建议额度的金额不良率下降比例=(原金额坏账率-按建议额度的金额坏账率率)/原金额坏账率

In [ ]:
data_end = data_all[data_all['行里实际审批结果'] == '通过']

In [ ]:
data_end['额度使用率'] = data_end['第6个mob实际账单透支余额'] / data_end['实际额度']
data_end['使用额度_实际'] = data_end['额度使用率'] * data_end['实际金额']
data_end['使用额度_建议'] = data_end['额度使用率'] * data_end['STR_LIMIT']
bad_rate_propose = data_end[data_end['label']==1]['使用额度_建议'].sum()/data_end['使用额度_建议'].sum()
bad_rate_init = data_end[data_end['label']==1]['使用额度_实际'].sum()/data_end['使用额度_实际'].sum()
print('按建议额度的金额坏账率: ',bad_rate_propose)
print('按实际额度的金额坏账率: ',bad_rate_init)
print('按建议额度的金额不良率下降比例: ',(bad_rate_init - bad_rate_propose)/bad_rate_init*100)

### 文件导出

In [ ]:
SAVE_PATH = FILE_PATH + '最终结果/'
isExists = os.path.exists(SAVE_PATH)
if not isExists:
    os.makedirs(SAVE_PATH) 

data_end[['APP_NO','STR_RESULT','STR_LIMIT','APP_SCORE']].to_csv(SAVE_PATH + '探知poc结果_贷前申请.csv', index=False)